# Initialize 

In [1]:
from pathlib import Path
import xarray as xr
import yaml
from tqdm.notebook import tqdm
import model_chain_inference as mci

In [2]:
path = Path("../data")
grid_data_file = path / "grid_data_flat.h5"
event_data_file = path / "event_data.h5"
fault_data_file = path / "fault_data.h5"

# Experiment settings

Define filters in space and time

In [3]:
experiment = "perf_1995_2025"

polygon_calibration = "GroningenFieldGWC"
mmin_calibration = 1.45
timeframe_calibration_prospective = ["1995-01-01", "2020-10-01"]
timeframe_calibration_retrospective = ["1995-01-01", "2025-10-01"]

polygon_etas = polygon_calibration
mmin_etas = mmin_calibration
timeframe_etas_prospective = ["1990-01-01", timeframe_calibration_prospective[1]]
timeframe_etas_retrospective = ["1990-01-01", timeframe_calibration_retrospective[1]]

In [4]:
etas_d = (2000.0) ** 2  # possibility to enter xarray
etas_q = 3.16  # possiblity to enter xarray

In [5]:
filterset = xr.Dataset(
    {
        "timeframe": xr.DataArray(
            data=[
                [
                    timeframe_calibration_prospective,
                    timeframe_etas_prospective,
                ],
                [
                    timeframe_calibration_retrospective,
                    timeframe_etas_retrospective,
                ],
            ],
            dims=["perspective", "purpose", "epoch"],
            coords={
                "perspective": ["prospective", "retrospective"],
                "purpose": ["calibration", "etas"],
                "epoch": ["start", "finish"],
            },
        ),
        "polygon": xr.DataArray(
            data=[
                polygon_calibration,
                polygon_etas,
            ],
            dims=["purpose"],
        ),
        "mmin": xr.DataArray(
            data=[
                mmin_calibration,
                mmin_etas,
            ],
            dims=["purpose"],
        ),
    }
)

# Import

In [6]:
event_data = xr.open_dataset(event_data_file, decode_coords="all").load()

In [7]:
fault_data = (
    xr.open_dataset(fault_data_file, decode_coords="all", decode_timedelta=False)
    .set_xindex(["bernstein_degree", "bernstein_index"])
    .set_xindex(["FAULT_ID", "PILLAR_ID"])
    .set_coords(["x", "y"])
)

In [8]:
grid_data_flat = (
    xr.open_dataset(
        grid_data_file, decode_coords="all", decode_timedelta=False, engine="h5netcdf"
    )
    .set_xindex(["x", "y"])
    .set_xindex(["bernstein_degree", "bernstein_index"])
)

# Data prep

## Define data scenarios

A data scenario (proper name pending) is combination of a covariate and a measure and possible some additional parameters.
The covariate is a predictor of the seismic event count/rate per unit of some measure. We need to choose both the covariate and the measure. An example of the covariate is stress, and examples of measures are reservoir volume, fault area, etc. 

In [9]:
data_scenarios = {
    "ETS_rmax": {
        "covariate_id": "stress_linear_rmax_smooth",
        "measure_id": "fault_area_rmax_smooth",
    },
    "EVTS_rmax": {
        "covariate_id": "stress_RTiCM_rmax_smooth",
        "measure_id": "fault_area_rmax_smooth",
    },
    "ETS_bs": {
        "covariate_id": "stress_linear_bs_smooth",
        "measure_id": "fault_area_bs_smooth",
    },
    "EVTS_bs": {
        "covariate_id": "stress_RTiCM_bs_smooth",
        "measure_id": "fault_area_bs_smooth",
    },
}

In [10]:
# Save scenarios yaml
with open(path / f"data_scenarios-{experiment}.yaml", "w") as f:
    yaml.dump(data_scenarios, f, default_flow_style=False)

## Prepare inference datasets

An inference dataset prepares the data of a data scenario into a compact format that is ready for inference.

In [11]:
for perspective in filterset["perspective"].values:
    run_id = f"{experiment}-{perspective}"
    print(f"--- Processing {run_id} ---")

    # Slice filterset
    filterset_run = filterset.sel(perspective=perspective)

    # Prepare kwargs
    purpose = "calibration"
    default_kwargs = {
        "event_data": event_data,
        "grid_data": grid_data_flat,
        "filterset": filterset_run.sel(purpose="calibration", drop=True),
        "filterset_etas": filterset_run.sel(purpose="etas", drop=True),
        "support_id": "support_fraction",
        "target_step": 0.02,
        "attributes": {"reservoir_thickness": 25.0},
        "etas_d": etas_d,
        "etas_q": etas_q,
    }

    # Generate data
    inference_data_dict = {}
    for k, d_kwargs in tqdm(data_scenarios.items(), desc=f"Scenarios {perspective}"):
        print(f"{k}")
        filepath = path / f"model_data-{run_id}-{k}.h5"

        # Check if file exists and load it if so
        if filepath.exists():
            print(f"File {filepath} already exists, skipping.")
        else:
            kwargs = default_kwargs | d_kwargs
            inf_data = mci.generate_inference_data_local(**kwargs)
            if "bernstein_basis" in inf_data.indexes:
                inf_data = inf_data.reset_index("bernstein_basis")

            # Save individual file
            inf_data.to_netcdf(filepath, engine="h5netcdf")

--- Processing perf_1995_2025-prospective ---


Scenarios prospective:   0%|          | 0/4 [00:00<?, ?it/s]

ETS_rmax
File ../data/model_data-perf_1995_2025-prospective-ETS_rmax.h5 already exists, skipping.
EVTS_rmax
File ../data/model_data-perf_1995_2025-prospective-EVTS_rmax.h5 already exists, skipping.
ETS_bs
File ../data/model_data-perf_1995_2025-prospective-ETS_bs.h5 already exists, skipping.
EVTS_bs
File ../data/model_data-perf_1995_2025-prospective-EVTS_bs.h5 already exists, skipping.
--- Processing perf_1995_2025-retrospective ---


Scenarios retrospective:   0%|          | 0/4 [00:00<?, ?it/s]

ETS_rmax
File ../data/model_data-perf_1995_2025-retrospective-ETS_rmax.h5 already exists, skipping.
EVTS_rmax
File ../data/model_data-perf_1995_2025-retrospective-EVTS_rmax.h5 already exists, skipping.
ETS_bs
File ../data/model_data-perf_1995_2025-retrospective-ETS_bs.h5 already exists, skipping.
EVTS_bs
File ../data/model_data-perf_1995_2025-retrospective-EVTS_bs.h5 already exists, skipping.
